# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbottabad123/flyrank-ml-track/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [12]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print("Token loaded:", "YES" if hf_token else "NO")

Token loaded: YES


The source data is at a daily content-page observation level: one row represents one client, one content page, and one report date. The available daily performance history spans 2025-01-27 through 2026-06-30. For the modelling task, features will be calculated from a historical window and the future performance window will be kept separate for defining the outcome.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields will include observable content and search-performance signals such as search volume, competition, word count, impressions, clicks, sessions, engagement, content age, update recency, and search position, provided they are available before the prediction window.

The label will be a future performance outcome defined from a later time window. Context fields such as content type and intent may be used for interpretation or controlled modelling.

Excluded fields include client names, domains, URLs, raw private queries, credentials, and any information that directly reveals or is derived from the future outcome. Future-window clicks, impressions, sessions, and trend measures will not be used as input features because they would create leakage.

In [13]:
from huggingface_hub import hf_hub_download
import duckdb

con = duckdb.connect()

dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=hf_token
)

dim_clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=hf_token
)

fact_march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

In [14]:
grain_check = con.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) as unique_combos
    FROM '{fact_march_path}'
""").df()
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_combos
0     9841378        9841378


In [15]:
date_span = con.execute(f"""
    SELECT
        COUNT(*) as row_count,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM '{fact_march_path}'
""").df()
print(date_span)

   row_count start_date   end_date
0    9841378 2026-03-01 2026-03-31


In [16]:
availability = con.execute(f"""
    SELECT
        COUNT(*) as rows_before,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_gsc,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_ga4
    FROM '{fact_march_path}'
""").df()
print(availability)

   rows_before  rows_with_gsc  rows_with_ga4
0      9841378      3611061.0       413966.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

# Load Hugging Face token
hf_token = userdata.get("HF_TOKEN")

print("Token loaded:", "YES" if hf_token else "NO")

# DuckDB connection
con = duckdb.connect()

# Download required warehouse files
dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=hf_token
)

dim_clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=hf_token
)

fact_march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

print("All required files loaded successfully.")


Token loaded: YES
All required files loaded successfully.


In [18]:
features = con.execute(f"""
    SELECT
        dc.content_hash_id,
        dc.client_hash_id,
        dc.search_volume,
        dc.competition,
        dc.backlinks,
        dc.word_count,
        AVG(f.gsc_avg_position) as avg_position_march
    FROM '{dim_content_path}' dc
    JOIN '{fact_march_path}' f
        ON dc.content_hash_id = f.content_hash_id
        AND dc.client_hash_id = f.client_hash_id
    WHERE dc.is_deleted IS FALSE
    GROUP BY dc.content_hash_id, dc.client_hash_id, dc.search_volume,
             dc.competition, dc.backlinks, dc.word_count
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,search_volume,competition,backlinks,word_count,avg_position_march
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,10,0.0,28,2999,5.147402
1,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,10,0.0,0,2855,5.145765
2,content_ac8663da7484669a,client_62f4a7e64f5e0096,10,0.0,0,3281,4.909314
3,content_39d7361b4945d504,client_62f4a7e64f5e0096,10,0.0,0,3579,4.074107
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,10,0.0,0,2455,4.428747


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [19]:
from scipy.stats import spearmanr
import numpy as np

features['honest_score'] = (
    features['search_volume'].rank()
    - features['competition'].rank()
    + features['avg_position_march'].rank()
)
features['honest_rank'] = features['honest_score'].rank(ascending=False)
features[['content_hash_id','honest_score','honest_rank']].head()

,content_hash_id,honest_score,honest_rank
0,content_d0dff76c889de68f,174826.0,47796.0
1,content_2e6360ad20fd7107,174799.0,47814.0
2,content_ac8663da7484669a,170627.0,50651.0
3,content_39d7361b4945d504,159712.0,59309.5
4,content_cec711b02f3bbde6,164076.0,55685.0


In [20]:
fact_april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    token=hf_token
)

future_leak = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_clicks) AS future_clicks
    FROM '{fact_april_path}'
    GROUP BY content_hash_id, client_hash_id
""").df()

features_leaked = features.merge(
    future_leak,
    on=['content_hash_id', 'client_hash_id'],
    how='left'
)

print("Rows with future-click data:",
      features_leaked['future_clicks'].notna().sum())

print("Total feature rows:",
      len(features_leaked))

Rows with future-click data: 324946
Total feature rows: 324947


In [21]:
features_final = features_leaked.drop(
    columns=['future_clicks']
)

print("Final feature rows:", len(features_final))
print("Future outcome removed:", 'future_clicks' not in features_final.columns)

features_final.head()

Final feature rows: 324947
Future outcome removed: True


,content_hash_id,client_hash_id,search_volume,competition,backlinks,word_count,avg_position_march,honest_score,honest_rank
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,10,0.0,28,2999,5.147402,174826.0,47796.0
1,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,10,0.0,0,2855,5.145765,174799.0,47814.0
2,content_ac8663da7484669a,client_62f4a7e64f5e0096,10,0.0,0,3281,4.909314,170627.0,50651.0
3,content_39d7361b4945d504,client_62f4a7e64f5e0096,10,0.0,0,3579,4.074107,159712.0,59309.5
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,10,0.0,0,2455,4.428747,164076.0,55685.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.